In [1]:
import torch

torch.cuda.is_available()

True

In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from tqdm import tqdm


pd.set_option('display.max_colwidth', None)


In [2]:

df_examples = pd.read_parquet('../data/shopping_queries_dataset_examples.parquet')
df_products = pd.read_parquet('../data/shopping_queries_dataset_products.parquet')

In [3]:
df_products.dropna(inplace=True)
# df_products.fillna(value='',inplace=True)

In [4]:
df_merged = df_products.merge(df_examples,how='inner',on=['product_id','product_locale'])

In [5]:
df_merged.isna().sum()

product_id              0
product_title           0
product_description     0
product_bullet_point    0
product_brand           0
product_color           0
product_locale          0
example_id              0
query                   0
query_id                0
esci_label              0
small_version           0
large_version           0
split                   0
dtype: int64

In [6]:
df_merged.head()

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,example_id,query,query_id,esci_label,small_version,large_version,split
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Negro, L","Esta playera con el logo de la marca Carrier de 11 Degrees viene en negro, con el logo de la marca en el pecho y un pequeño texto en la parte posterior. La camiseta tiene cuello redondo y manga corta.",11 Degrees Negro Playera con logo\nA estrenar y genuina. Somos un vendedor autorizado de 11 Degrees.\nVer descripción del producto para obtener más información.,11 Degrees,Negro,es,47092,11 degrees,1688,E,0,1,train
1,B07DP4LM9H,"11 Degrees de los Hombres Core Pull Over Hoodie, Azul, S","La sudadera con capucha Core Pull Over de 11 Grados viene en color azul marino, con una capucha ajustable con cordones. Con un bolsillo en la parte delantera, esta sudadera con capucha para hombre tiene un logotipo de goma en la parte delantera y ojales de marca.",11 Degrees Azul Core Pull Over Hoodie\nA estrenar y genuina. Somos un vendedor autorizado de 11 Degrees.\nVer descripción del producto para obtener más información.,11 Degrees,Azul,es,47089,11 degrees,1688,E,0,1,train
2,B07MSD1JH3,"11 Degrees de los Hombres Optum Poly Joggers, Negro, XL","Los Optum Poly Joggers de 11 grados vienen con tobillos elásticos con cremallera y un logotipo de goma en el muslo. En color negro, estos joggers para hombre cuentan con cordón elástico en la cintura y bolsillos laterales abiertos. Estos corredores también se divierten marcando ambas piernas.",11 Degrees Negro Optum Poly Joggers\nA estrenar y genuina. Somos un vendedor autorizado de 11 Degrees.\nVer descripción del producto para obtener más información.,11 Degrees,Negro,es,47086,11 degrees,1688,E,0,1,train
3,B07VCV1LSQ,11 Degrees Chaqueta Espacial S Black,"La chaqueta Space Puffer de 11 Degrees viene con bolsillos con cremallera, con un logotipo de goma en el pecho. Esta chaqueta, que viene en color negro, tiene un cierre de cremallera y una capucha elástica. Esta chaqueta también tiene puños y dobladillo elásticos.",11 Degrees Negro Chaqueta acolchada\nA estrenar y genuina. Somos un vendedor autorizado de 11 Degrees.\nVer descripción del producto para obtener más información.,11 Degrees,Negro,es,47082,11 degrees,1688,E,0,1,train
4,B07VQVZYYS,11 Degrees Chaqueta Espacial S Blk Wht,"La chaqueta Space Puffer de 11 Degrees viene en color negro / blanco, con cierre de cremallera y capucha elástica. Esta chaqueta también tiene puños y dobladillo elásticos. Con bolsillos con cremallera, esta chaqueta tiene un logotipo impreso en el pecho.",11 Degrees Negro Chaqueta acolchada\nA estrenar y genuina. Somos un vendedor autorizado de 11 Degrees.\nVer descripción del producto para obtener más información.,11 Degrees,Negro,es,47081,11 degrees,1688,E,0,1,train


In [7]:
df_merged = df_merged[df_merged['product_locale'] == 'us']

In [8]:
df_merged.shape

(632018, 14)

In [9]:
df_merged.drop(labels=['query_id','large_version','small_version','example_id','product_locale'],axis=1,inplace=True)

In [10]:
df_merged.drop_duplicates(subset=['product_id'], inplace=True)

In [11]:
df_merged.shape

(437964, 9)

In [13]:
df = df_merged.sample(n=100000,random_state=42)

In [14]:
df.shape

(100000, 9)

In [15]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    return text.lower().strip()  # Convert to lowercase

# Apply cleaning to each column
columns_to_clean = ['product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color']
df[columns_to_clean] = df[columns_to_clean].applymap(clean_text)

/tmp/ipykernel_90980/4043134069.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[columns_to_clean] = df[columns_to_clean].applymap(clean_text)


In [16]:
df['binary_label'] = df['esci_label'].apply(lambda x: 1 if x == 'E' else 0)

In [17]:
df.replace('', np.nan, inplace=True)

In [20]:
df.isnull().sum()

product_id              0
product_title           0
product_description     0
product_bullet_point    0
product_brand           0
product_color           0
query                   0
esci_label              0
split                   0
binary_label            0
dtype: int64

In [19]:
df.dropna(inplace=True)

In [21]:
df.to_csv('../data/data.csv',index=False)

In [2]:
df = pd.read_csv('../data/data.csv')

In [3]:
df.isnull().sum()

product_id              0
product_title           0
product_description     0
product_bullet_point    0
product_brand           0
product_color           0
query                   0
esci_label              0
split                   0
binary_label            0
dtype: int64

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base',trust_remote_code=True)


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: {'classifier.weight', 'classifier.bias'}
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


- make embeddings of all features
- each feature will have 1024 dimensional embedding

In [5]:
def get_embeddings(text):
    embeddings = model.encode(text, normalize_embeddings=True, show_progress_bar=True)
    return embeddings.squeeze()


In [10]:
df.isnull().sum()

product_id              0
product_title           0
product_description     0
product_bullet_point    0
product_brand           0
product_color           0
query                   0
esci_label              0
split                   0
binary_label            0
dtype: int64

In [6]:
columns = ['product_title',
           'product_description',
           'product_bullet_point', 
           'product_brand', 
           'product_color',
           'query']
for col in columns:
    print(f'{col}:')
    np.save(f'../data/new_embeddings/{col}_embedding',get_embeddings(df[col].tolist()))

product_title:


Batches:   0%|          | 0/3123 [00:00<?, ?it/s]

product_description:


Batches:   0%|          | 0/3123 [00:00<?, ?it/s]

product_bullet_point:


...

Batches:   0%|          | 0/3123 [00:00<?, ?it/s]

product_color:


Batches:   0%|          | 0/3123 [00:00<?, ?it/s]

query:


Batches:   0%|          | 0/3123 [00:00<?, ?it/s]

In [17]:
emb = np.load('../data/bert_embeddings/product_description_embedding.npy')

In [ ]:
np.column_stack((df.index, df['product_id'].values))

array([[208182, 'B076MSCX16'],
       [67217, 'B074H2MQHS'],
       [686469, 'B081R4WZDZ'],
       ...,
       [84153, 'B00KV5A3QK'],
       [124987, 'B07C3LZJLZ'],
       [708494, 'B07TTZ37KH']], shape=(100000, 2), dtype=object)

In [28]:
np.save('key',np.column_stack((df.index, df['product_id'].values)))

In [31]:
df['binary_label'] = df['esci_label'].apply(lambda x: 1 if x == 'E' else 0)

In [32]:
df.to_csv('data.csv')

ESCI relevance judgements (Exact, Substitute, Complement, Irrelevant) 

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data.csv')

In [3]:
df.drop(labels=['Unnamed: 0'],inplace=True, axis=1)

In [4]:
df['esci_label'].value_counts()

esci_label
E    71850
S    18570
I     7297
C     2283
Name: count, dtype: int64

In [5]:
df['split'].value_counts()

split
train    76728
test     23272
Name: count, dtype: int64

In [6]:
df['binary_label'] = df['esci_label'].apply(lambda x: 1 if x == 'E' else 0)

In [7]:
df['binary_label'].value_counts()

binary_label
1    71850
0    28150
Name: count, dtype: int64

In [8]:
pd.set_option('display.max_columns', 200)

In [9]:
df['product_brand_embedding'][0]

'[-6.68430775e-02  5.84305823e-02 -2.26592235e-02 -2.25061625e-02\n  2.04019174e-02 -4.88883890e-02  5.78287430e-03  1.09870406e-03\n  2.17257254e-02 -3.26123945e-02 -1.26661599e-01 -4.30237092e-02\n -1.21828668e-01  8.67680181e-03 -6.49383366e-02  4.24251519e-02\n  1.15383908e-01  8.40374082e-02  2.79561486e-02  9.22133699e-02\n  6.87521920e-02  3.04094912e-03 -2.09577996e-02 -1.08269705e-02\n -4.39583249e-02  1.31887048e-01  1.52454656e-02 -1.03353383e-02\n -7.67243803e-02  7.66213611e-02  9.33080446e-03 -2.46249344e-02\n  6.42584413e-02  5.89809893e-03  4.97500412e-02 -4.84317839e-02\n  2.77336538e-02  2.97054881e-03 -3.33276428e-02 -1.08872000e-02\n -5.84141053e-02 -2.07679067e-02  2.44317204e-02  6.26201630e-02\n  7.28961229e-02  5.66222593e-02 -2.74797827e-02  5.52076697e-02\n -4.32891957e-02  5.33125997e-02 -3.33009250e-02  4.18664403e-02\n  8.57473630e-03  6.23890385e-03  6.08814124e-04  6.38413895e-03\n -8.80445987e-02  5.46693914e-02  5.20456582e-02  8.72146059e-03\n  5.38239

In [11]:
columns = ['product_title_embedding',
           'product_description_embedding',
           'product_bullet_point_embedding', 
           'product_brand_embedding', 
           'product_color_embedding',
           'query_embedding']

for col in columns:
    # df[col] = df[col].str.replace('\n ','')
    df[col] = df[col].str.replace('  ',' ')


In [12]:
df['product_brand_embedding'][0]

'[-6.68430775e-02 5.84305823e-02 -2.26592235e-02 -2.25061625e-02 2.04019174e-02 -4.88883890e-02 5.78287430e-03 1.09870406e-03 2.17257254e-02 -3.26123945e-02 -1.26661599e-01 -4.30237092e-02-1.21828668e-01 8.67680181e-03 -6.49383366e-02 4.24251519e-02 1.15383908e-01 8.40374082e-02 2.79561486e-02 9.22133699e-02 6.87521920e-02 3.04094912e-03 -2.09577996e-02 -1.08269705e-02-4.39583249e-02 1.31887048e-01 1.52454656e-02 -1.03353383e-02-7.67243803e-02 7.66213611e-02 9.33080446e-03 -2.46249344e-02 6.42584413e-02 5.89809893e-03 4.97500412e-02 -4.84317839e-02 2.77336538e-02 2.97054881e-03 -3.33276428e-02 -1.08872000e-02-5.84141053e-02 -2.07679067e-02 2.44317204e-02 6.26201630e-02 7.28961229e-02 5.66222593e-02 -2.74797827e-02 5.52076697e-02-4.32891957e-02 5.33125997e-02 -3.33009250e-02 4.18664403e-02 8.57473630e-03 6.23890385e-03 6.08814124e-04 6.38413895e-03-8.80445987e-02 5.46693914e-02 5.20456582e-02 8.72146059e-03 5.38239516e-02 -3.39882355e-03 -1.63745563e-02 -3.64664569e-02-5.32084331e-02 -2

In [ ]:
columns = ['product_title_embedding',
           'product_description_embedding',
           'product_bullet_point_embedding', 
           'product_brand_embedding', 
           'product_color_embedding',
           'query_embedding']
for col in columns:
    
    array = np.fromstring(s[1:-1], sep=" ")

In [5]:
encoding_map = {'E': 1, 'I': 0, 'S': 2, 'C': 3}

# Apply encoding
df['encoded_label'] = df['esci_label'].map(encoding_map)

In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TwoTower(nn.Module):
    def __init__(self, product_embed_dim, query_embed_dim, hidden_dim, output_size):
        super().__init__()
        self.product_tower_layer1 = nn.Linear(product_embed_dim, hidden_dim)  # First hidden layer
        self.product_tower_layer2 = nn.Linear(hidden_dim, output_size)
        self.query_tower_layer1 = nn.Linear(query_embed_dim, hidden_dim)     # Second hidden layer
        self.query_tower_layer2 = nn.Linear(hidden_dim, output_size)
        self.cos = nn.CosineSimilarity(dim=-1)
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()                

    def forward(self, product_embed, query_embed):
        x_p = self.product_tower_layer1(product_embed)  # Apply ReLU after first layer
        x_p = self.relu(x_p)
        x_p = self.product_tower_layer2(x_p)

        x_q = self.query_tower_layer1(query_embed)  # Apply ReLU after second layer
        x_q = self.relu(x_q)
        x_q = self.query_tower_layer2(x_q)
        
        x_p = F.normalize(x_p, p=2, dim=-1)
        x_q = F.normalize(x_q, p=2, dim=-1)
        output = self.sigmoid(self.cos(x_p, x_q))


        return output

model = TwoTower(product_embed_dim=1000,query_embed_dim=1000, hidden_dim=512, output_size=32)

print(model)

TwoTower(
  (product_tower_layer1): Linear(in_features=1000, out_features=512, bias=True)
  (product_tower_layer2): Linear(in_features=512, out_features=32, bias=True)
  (query_tower_layer1): Linear(in_features=1000, out_features=512, bias=True)
  (query_tower_layer2): Linear(in_features=512, out_features=32, bias=True)
  (cos): CosineSimilarity()
  (sigmoid): Sigmoid()
  (relu): ReLU()
)


In [39]:
p = torch.rand(1000)  # Adjust size to match input_size
q = torch.rand(1000)

output = model(p,q)

print("Output:", output)

Output: tensor(0.5260, grad_fn=<SigmoidBackward0>)
